# 14 — Election Results to OA21 Crosswalk

This notebook allocates source-year ward/division result summaries to OA21 rows.

It uses:

- `ward_result_summary_v1.csv` or `ward_result_summary_v2_with_2023_matches.csv`
- `election_geography_join_readiness_v1.csv`
- OA21→WD22/23/24/25 lookup files
- WD→CED bridge files where needed
- the K7 OA base file for OA population

The output is an OA-level allocated election file suitable for joining to the K7 atlas in Notebook 15.

2021 rows are deliberately excluded because the available 2021 lookup is OA11-based, not OA21-based.


## 14.1 Project paths and input files


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

ELECTION_DIR = PROJECT_DIR / "data" / "processed" / "election_results"
READINESS_DIR = ELECTION_DIR / "geography_readiness_v1"
GEOGRAPHY_DIR = PROJECT_DIR / "data" / "geography"
AGG_DIR = PROJECT_DIR / "data" / "processed" / "aggregations_v1"
ATLAS_DIR = PROJECT_DIR / "data" / "processed" / "atlas_outputs_v1"
OUTPUT_DIR = ELECTION_DIR / "oa21_allocated_v1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WARD_SUMMARY_PATH = ELECTION_DIR / "ward_result_summary_v2_with_2023_matches.csv"
READINESS_PATH = READINESS_DIR / "election_geography_join_readiness_v1.csv"

# Prefer atlas_outputs_v1 if it exists, otherwise use aggregations_v1.
OA_BASE_CANDIDATES = [
    ATLAS_DIR / "k7_oa_geo_cluster_base_v1.csv",
    AGG_DIR / "k7_oa_geo_cluster_base_v1.csv",
]

OA_BASE_PATH = next((p for p in OA_BASE_CANDIDATES if p.exists()), None)

if OA_BASE_PATH is None:
    raise FileNotFoundError("Could not find k7_oa_geo_cluster_base_v1.csv in atlas_outputs_v1 or aggregations_v1.")

GEOGRAPHY_FILES = {
    2022: {
        "oa_to_ward": "oa21_to_wd22_lad22_ctyua22_rgn22_ctry22_eng_wal.csv",
        "wd_to_ced": "wd22_to_lad22_cty22_ced22_eng.csv",
    },
    2023: {
        "oa_to_ward": "oa21_to_wd23_lad23_eng_wal.csv",
        "wd_to_ced": "wd23_to_lad23_cty23_ced23_eng.csv",
    },
    2024: {
        "oa_to_ward": "oa21_to_wd24_lad24_eng_wal.csv",
        "wd_to_ced": "wd24_to_lad24_cty24_ced24_eng.csv",
    },
    2025: {
        "oa_to_ward": "oa21_to_wd25_lad25_eng_wal(may25).csv",
        "wd_to_ced": "wd25_to_lad25_cty25_ced25_eng.csv",
    },
}

print("Project directory:", PROJECT_DIR)
print("Election directory:", ELECTION_DIR)
print("Geography directory:", GEOGRAPHY_DIR)
print("OA base path:", OA_BASE_PATH)
print("Output directory:", OUTPUT_DIR)

for path in [WARD_SUMMARY_PATH, READINESS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")


Project directory: c:\Users\keena\Documents\Electoral_Tribes
Election directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results
Geography directory: c:\Users\keena\Documents\Electoral_Tribes\data\geography
OA base path: c:\Users\keena\Documents\Electoral_Tribes\data\processed\aggregations_v1\k7_oa_geo_cluster_base_v1.csv
Output directory: c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\oa21_allocated_v1


## 14.2 Helper functions


In [13]:
def clean_colname(col):
    return str(col).strip()


def normalise_code(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    if value in ["", "NAN", "NONE", "NULL"]:
        return pd.NA
    return value


def read_lookup_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Lookup file not found: {path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path, low_memory=False)

    df.columns = [clean_colname(c) for c in df.columns]
    return df


def find_first_col(df, candidates):
    cols = {c.upper(): c for c in df.columns}
    for cand in candidates:
        if cand.upper() in cols:
            return cols[cand.upper()]
    return None


def standardise_oa_to_ward_lookup(path, year):
    df = read_lookup_file(path)
    yy = str(year)[-2:]

    oa_col = find_first_col(df, ["OA21CD", "OA21 code", "OA code"])
    wd_col = find_first_col(df, [f"WD{yy}CD", "WDCD", "Ward code", "Ward/ED code"])
    wd_name_col = find_first_col(df, [f"WD{yy}NM", "WDNM", "Ward name", "Ward/ED name"])
    lad_col = find_first_col(df, [f"LAD{yy}CD", "LADCD", "Local authority code"])
    lad_name_col = find_first_col(df, [f"LAD{yy}NM", "LADNM", "Local authority name"])

    required = {"OA21CD": oa_col, "source_ward_code": wd_col}
    missing = [name for name, col in required.items() if col is None]
    if missing:
        raise ValueError(f"{path.name} missing required columns: {missing}. Columns: {df.columns.tolist()}")

    out = pd.DataFrame({
        "OA21CD": df[oa_col].map(normalise_code),
        "source_boundary_year": year,
        "source_ward_code": df[wd_col].map(normalise_code),
        "source_ward_name": df[wd_name_col] if wd_name_col else pd.NA,
        "source_lad_code": df[lad_col].map(normalise_code) if lad_col else pd.NA,
        "source_lad_name": df[lad_name_col] if lad_name_col else pd.NA,
    })

    out = out.dropna(subset=["OA21CD", "source_ward_code"]).drop_duplicates()
    return out


def standardise_wd_to_ced_lookup(path, year):
    df = read_lookup_file(path)
    yy = str(year)[-2:]

    wd_col = find_first_col(df, [f"WD{yy}CD", "WDCD", "Ward code", "Ward/ED code"])
    wd_name_col = find_first_col(df, [f"WD{yy}NM", "WDNM", "Ward name", "Ward/ED name"])
    ced_col = find_first_col(df, [f"CED{yy}CD", "CEDCD", "County electoral division code"])
    ced_name_col = find_first_col(df, [f"CED{yy}NM", "CEDNM", "County electoral division name"])

    if wd_col is None or ced_col is None:
        raise ValueError(f"{path.name} must contain ward and CED code columns. Columns: {df.columns.tolist()}")

    out = pd.DataFrame({
        "source_boundary_year": year,
        "source_ward_code": df[wd_col].map(normalise_code),
        "source_ward_name": df[wd_name_col] if wd_name_col else pd.NA,
        "source_ced_code": df[ced_col].map(normalise_code),
        "source_ced_name": df[ced_name_col] if ced_name_col else pd.NA,
    })

    out = out.dropna(subset=["source_ward_code", "source_ced_code"]).drop_duplicates()
    return out


def make_result_area_id(row):
    return row.get("result_area_key")


## 14.3 Load inputs


In [14]:
ward_summary = pd.read_csv(WARD_SUMMARY_PATH, low_memory=False)
readiness = pd.read_csv(READINESS_PATH, low_memory=False)
oa_base = pd.read_csv(OA_BASE_PATH, low_memory=False)

for df in [ward_summary, readiness, oa_base]:
    if "OA21CD" in df.columns:
        df["OA21CD"] = df["OA21CD"].map(normalise_code)

# Authoritative OA population and current K7 atlas geography.
oa_pop = oa_base[["OA21CD", "population"]].drop_duplicates("OA21CD").copy()
oa_pop["population"] = pd.to_numeric(oa_pop["population"], errors="coerce")

print("Ward summary rows:", len(ward_summary))
print("Readiness rows:", len(readiness))
print("OA base rows:", len(oa_base))
print("OA population total:", oa_pop["population"].sum())

display(readiness["join_readiness_status"].value_counts(dropna=False))


Ward summary rows: 15609
Readiness rows: 15609
OA base rows: 188880
OA population total: 59597747


join_readiness_status
ward_oa21_lookup_available            10856
not_oa21_ready_oa11_lookup_only        3864
ced_bridge_available                    888
ward_code_not_found_in_oa21_lookup        1
Name: count, dtype: int64

## 14.4 Build source-year OA crosswalks

For ordinary ward/electoral-division rows, this uses OA21→WDxx.

For county electoral divisions, this uses OA21→WDxx plus WDxx→CEDxx as a bridge. This is an approximation and should be treated as lower confidence than direct ward lookups.


In [15]:
ward_crosswalks = []
ced_crosswalks = []
ced_ambiguity_reports = []

for year, spec in GEOGRAPHY_FILES.items():
    print(f"\nProcessing geography crosswalks for {year}...")

    oa_to_ward = standardise_oa_to_ward_lookup(
        GEOGRAPHY_DIR / spec["oa_to_ward"],
        year
    )

    oa_to_ward["crosswalk_type"] = "oa21_to_source_ward"
    ward_crosswalks.append(oa_to_ward)

    # CED bridge: OA -> source ward -> source CED.
    #
    # Important:
    # We can only use this bridge safely where each source ward maps to exactly
    # one county electoral division. If a ward maps to multiple CEDs, we cannot
    # allocate OA rows safely from this lookup alone.
    wd_to_ced_path = GEOGRAPHY_DIR / spec["wd_to_ced"]

    if not wd_to_ced_path.exists():
        print(f"  No WD→CED lookup found for {year}: {wd_to_ced_path.name}")
        continue

    wd_to_ced = standardise_wd_to_ced_lookup(wd_to_ced_path, year)

    required = [
        "source_boundary_year",
        "source_ward_code",
        "source_ced_code",
        "source_ced_name",
    ]

    missing = [c for c in required if c not in wd_to_ced.columns]
    if missing:
        raise ValueError(f"{wd_to_ced_path.name} missing required columns: {missing}")

    # Remove exact duplicate rows first.
    wd_to_ced_clean = (
        wd_to_ced[required]
        .dropna(subset=["source_ward_code", "source_ced_code"])
        .drop_duplicates()
        .copy()
    )

    # Count how many distinct CEDs each ward maps to.
    ward_to_ced_counts = (
        wd_to_ced_clean
        .groupby(["source_boundary_year", "source_ward_code"], as_index=False)
        .agg(
            ced_count=("source_ced_code", "nunique"),
            ced_codes=("source_ced_code", lambda x: "; ".join(sorted(set(map(str, x))))),
            ced_names=("source_ced_name", lambda x: "; ".join(sorted(set(map(str, x)))))
        )
    )

    unique_ward_keys = ward_to_ced_counts.query("ced_count == 1")[
        ["source_boundary_year", "source_ward_code"]
    ]

    ambiguous_ward_keys = ward_to_ced_counts.query("ced_count > 1").copy()
    ambiguous_ward_keys["year"] = year

    print(f"  WD→CED rows after exact dedupe: {len(wd_to_ced_clean):,}")
    print(f"  Wards with unique CED mapping: {len(unique_ward_keys):,}")
    print(f"  Wards with ambiguous CED mapping: {len(ambiguous_ward_keys):,}")

    if len(ambiguous_ward_keys) > 0:
        ced_ambiguity_reports.append(ambiguous_ward_keys)

    # Keep only safe one-ward-to-one-CED mappings.
    wd_to_ced_unique = wd_to_ced_clean.merge(
        unique_ward_keys,
        on=["source_boundary_year", "source_ward_code"],
        how="inner",
        validate="many_to_one"
    )

    # Now this merge is safe because the right side has one CED per ward.
    ced_bridge = oa_to_ward.merge(
        wd_to_ced_unique,
        on=["source_boundary_year", "source_ward_code"],
        how="inner",
        validate="many_to_one"
    )

    ced_bridge = ced_bridge.rename(columns={
        "source_ced_code": "source_division_code",
        "source_ced_name": "source_division_name",
    })

    ced_bridge["crosswalk_type"] = "oa21_to_source_ced_via_unique_ward_bridge"
    ced_bridge["ced_bridge_confidence"] = "medium_unique_ward_to_ced"

    ced_crosswalks.append(ced_bridge)

ward_crosswalk = pd.concat(ward_crosswalks, ignore_index=True)

if ced_crosswalks:
    ced_crosswalk = pd.concat(ced_crosswalks, ignore_index=True)
else:
    ced_crosswalk = pd.DataFrame()

ward_crosswalk = ward_crosswalk.merge(
    oa_pop,
    on="OA21CD",
    how="left",
    validate="many_to_one"
)

if len(ced_crosswalk) > 0:
    ced_crosswalk = ced_crosswalk.merge(
        oa_pop,
        on="OA21CD",
        how="left",
        validate="many_to_one"
    )

print("\nFinal crosswalk row counts")
print("Ward crosswalk rows:", len(ward_crosswalk))
print("CED crosswalk rows:", len(ced_crosswalk))

display(ward_crosswalk.head())

if len(ced_crosswalk) > 0:
    display(ced_crosswalk.head())

ward_crosswalk.to_csv(
    OUTPUT_DIR / "source_ward_to_oa21_crosswalk_v1.csv",
    index=False
)

if len(ced_crosswalk) > 0:
    ced_crosswalk.to_csv(
        OUTPUT_DIR / "source_ced_to_oa21_crosswalk_v1.csv",
        index=False
    )

if ced_ambiguity_reports:
    ced_ambiguity_report = pd.concat(ced_ambiguity_reports, ignore_index=True)

    ced_ambiguity_report.to_csv(
        OUTPUT_DIR / "ced_ward_bridge_ambiguity_report_v1.csv",
        index=False
    )

    print("\nCED ambiguity report saved:")
    print(OUTPUT_DIR / "ced_ward_bridge_ambiguity_report_v1.csv")

    display(ced_ambiguity_report.head(20))
else:
    print("\nNo ambiguous WD→CED mappings found.")


Processing geography crosswalks for 2022...
  WD→CED rows after exact dedupe: 4,950
  Wards with unique CED mapping: 2,525
  Wards with ambiguous CED mapping: 1,116

Processing geography crosswalks for 2023...
  WD→CED rows after exact dedupe: 4,695
  Wards with unique CED mapping: 2,176
  Wards with ambiguous CED mapping: 1,155

Processing geography crosswalks for 2024...
  WD→CED rows after exact dedupe: 4,759
  Wards with unique CED mapping: 2,062
  Wards with ambiguous CED mapping: 1,233

Processing geography crosswalks for 2025...
  WD→CED rows after exact dedupe: 4,654
  Wards with unique CED mapping: 2,123
  Wards with ambiguous CED mapping: 1,170

Final crosswalk row counts
Ward crosswalk rows: 755520
CED crosswalk rows: 155243


,OA21CD,source_boundary_year,source_ward_code,source_ward_name,source_lad_code,source_lad_name,crosswalk_type,population
0,E00000001,2022,E09000001,City of London,<NA>,<NA>,oa21_to_source_ward,176
1,E00000003,2022,E09000001,City of London,<NA>,<NA>,oa21_to_source_ward,256
2,E00000005,2022,E09000001,City of London,<NA>,<NA>,oa21_to_source_ward,112
3,E00000007,2022,E09000001,City of London,<NA>,<NA>,oa21_to_source_ward,144
4,E00000010,2022,E09000001,City of London,<NA>,<NA>,oa21_to_source_ward,178


,OA21CD,source_boundary_year,source_ward_code,source_ward_name,source_lad_code,source_lad_name,crosswalk_type,source_division_code,source_division_name,ced_bridge_confidence,population
0,E00090874,2022,E05011561,Burwell,<NA>,<NA>,oa21_to_source_ced_via_unique_ward_bridge,E58000055,Burwell ED,medium_unique_ward_to_ced,248
1,E00090875,2022,E05011561,Burwell,<NA>,<NA>,oa21_to_source_ced_via_unique_ward_bridge,E58000055,Burwell ED,medium_unique_ward_to_ced,307
2,E00090876,2022,E05011561,Burwell,<NA>,<NA>,oa21_to_source_ced_via_unique_ward_bridge,E58000055,Burwell ED,medium_unique_ward_to_ced,357
3,E00090877,2022,E05011561,Burwell,<NA>,<NA>,oa21_to_source_ced_via_unique_ward_bridge,E58000055,Burwell ED,medium_unique_ward_to_ced,332
4,E00090878,2022,E05011561,Burwell,<NA>,<NA>,oa21_to_source_ced_via_unique_ward_bridge,E58000055,Burwell ED,medium_unique_ward_to_ced,269



CED ambiguity report saved:
c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\oa21_allocated_v1\ced_ward_bridge_ambiguity_report_v1.csv


,source_boundary_year,source_ward_code,ced_count,ced_codes,ced_names,year
0,2022,E05003286,2,E58000222; E58000235,Heanor Central ED; Ripley East and Codnor ED,2022
1,2022,E05003300,2,E58000194; E58000216,Alport and Derwent ED; Duffield and Belper Sou...,2022
2,2022,E05003326,2,E58000203; E58000230,Boythorpe and Brampton South ED; Loundsley Gre...,2022
3,2022,E05003328,2,E58000200; E58000241,Birdholme ED; Spire ED,2022
4,2022,E05003329,2,E58000206; E58000243,Brimington ED; Staveley ED,2022
5,2022,E05003331,2,E58000230; E58000242,Loundsley Green and Newbold ED; St. Mary's ED,2022
6,2022,E05003333,2,E58000243; E58000244,Staveley ED; Staveley North and Whittington ED,2022
7,2022,E05003337,2,E58000200; E58000203,Birdholme ED; Boythorpe and Brampton South ED,2022
8,2022,E05003338,2,E58000230; E58000241,Loundsley Green and Newbold ED; Spire ED,2022
9,2022,E05003339,2,E58000200; E58000241,Birdholme ED; Spire ED,2022


## 14.5 Select result areas ready for OA21 allocation


In [16]:
ready = readiness[readiness["atlas_join_ready"].astype(bool)].copy()
ready["source_boundary_year"] = pd.to_numeric(ready["source_boundary_year"], errors="coerce").astype("Int64")
ready["source_geography_code"] = ready["source_geography_code"].map(normalise_code)

# Keep one row per result area.
ready_areas = ready.drop_duplicates("result_area_key", keep="first").copy()

print("Ready result areas:", len(ready_areas))
display(ready_areas[["source_year", "source_boundary_year", "detected_geography_type", "join_readiness_status"]].value_counts().reset_index(name="n"))


Ready result areas: 11744


,source_year,source_boundary_year,detected_geography_type,join_readiness_status,n
0,2023,2023,electoral_ward_or_division,ward_oa21_lookup_available,4831
1,2022,2022,electoral_ward_or_division,ward_oa21_lookup_available,3609
2,2024,2024,electoral_ward_or_division,ward_oa21_lookup_available,1903
3,2025,2025,county_electoral_division,ced_bridge_available,888
4,2025,2025,electoral_ward_or_division,ward_oa21_lookup_available,513


## 14.6 Allocate ward/division results to OA21


In [17]:
# ============================================================
# 14.6 Allocate ready election result areas to OA21
# ============================================================
#
# This cell takes source-year ward/division result summaries and allocates
# their numeric election values to OA21 using population weights.
#
# Important:
# - The same source ward code may appear more than once in the election data.
#   That is not automatically an error.
# - The true unique election unit is result_area_key.
# - Allocation weights must therefore be calculated within result_area_key,
#   not simply within source_geography_code.
# ============================================================

# Numeric columns that should be allocated by population weight.
base_numeric_cols = [
    "electorate",
    "valid_votes",
    "ballots",
    "invalid_votes",
    "top_party_votes",
    "runner_up_party_votes",
    "con_votes",
    "lab_votes",
    "ld_votes",
    "green_votes",
    "reform_ukip_brexit_votes",
    "independent_votes",
    "sdp_votes",
    "other_votes",
]

numeric_cols = [c for c in base_numeric_cols if c in ready_areas.columns]

for col in numeric_cols:
    ready_areas[col] = pd.to_numeric(ready_areas[col], errors="coerce")

# Basic sanity check: result_area_key should be unique in the ward summary.
duplicate_result_keys = (
    ready_areas
    .groupby("result_area_key", as_index=False)
    .size()
    .query("size > 1")
)

print("Ready result areas:", ready_areas["result_area_key"].nunique())
print("Ready rows:", len(ready_areas))
print("Duplicate result_area_key rows:", len(duplicate_result_keys))

if len(duplicate_result_keys) > 0:
    display(duplicate_result_keys.head(20))
    raise ValueError(
        "ready_areas contains duplicate result_area_key values. "
        "Fix ward_result_summary_v1.csv before allocation."
    )

# Diagnostic: source ward/division codes may repeat. This is allowed.
source_code_reuse = (
    ready_areas
    .groupby(["source_boundary_year", "source_geography_code"], as_index=False)
    .agg(
        result_area_count=("result_area_key", "nunique"),
        rows=("result_area_key", "size"),
    )
    .query("result_area_count > 1")
    .sort_values(["source_boundary_year", "result_area_count"], ascending=[True, False])
)

print("Repeated source geography codes:", len(source_code_reuse))
if len(source_code_reuse) > 0:
    display(source_code_reuse.head(20))

# ------------------------------------------------------------
# Ward / electoral ward allocation
# ------------------------------------------------------------

ward_ready = ready_areas[
    ready_areas["detected_geography_type"].eq("electoral_ward_or_division")
].copy()

print("Ward-ready result areas:", ward_ready["result_area_key"].nunique())

ward_alloc = ward_ready.merge(
    ward_crosswalk,
    left_on=["source_boundary_year", "source_geography_code"],
    right_on=["source_boundary_year", "source_ward_code"],
    how="inner",
    validate="many_to_many"
)

ward_alloc["allocation_method"] = "source_ward_to_oa21_population_weighted"
ward_alloc["source_area_code_for_allocation"] = ward_alloc["source_ward_code"]

# ------------------------------------------------------------
# County electoral division allocation
# ------------------------------------------------------------

ced_ready = ready_areas[
    ready_areas["detected_geography_type"].eq("county_electoral_division")
].copy()

print("CED-ready result areas:", ced_ready["result_area_key"].nunique())

if len(ced_ready) > 0 and len(ced_crosswalk) > 0:
    ced_alloc = ced_ready.merge(
        ced_crosswalk,
        left_on=["source_boundary_year", "source_geography_code"],
        right_on=["source_boundary_year", "source_division_code"],
        how="inner",
        validate="many_to_many"
    )

    ced_alloc["allocation_method"] = "source_ced_to_oa21_via_ward_population_weighted"
    ced_alloc["source_area_code_for_allocation"] = ced_alloc["source_division_code"]
else:
    ced_alloc = pd.DataFrame(columns=ward_alloc.columns)

# ------------------------------------------------------------
# Combine allocations
# ------------------------------------------------------------

allocated = pd.concat([ward_alloc, ced_alloc], ignore_index=True, sort=False)

allocated["population"] = pd.to_numeric(allocated["population"], errors="coerce")

# The population base must be calculated per election result area.
allocated["source_area_population"] = (
    allocated
    .groupby("result_area_key")["population"]
    .transform("sum")
)

allocated["allocation_weight"] = np.where(
    allocated["source_area_population"] > 0,
    allocated["population"] / allocated["source_area_population"],
    np.nan
)

# Allocate numeric election values to OAs.
for col in numeric_cols:
    allocated[f"allocated_{col}"] = allocated[col] * allocated["allocation_weight"]

# ------------------------------------------------------------
# Allocation diagnostics
# ------------------------------------------------------------

allocation_check = (
    allocated
    .groupby("result_area_key", as_index=False)
    .agg(
        source_boundary_year=("source_boundary_year", "first"),
        detected_geography_type=("detected_geography_type", "first"),
        source_geography_code=("source_geography_code", "first"),
        source_geography_name=("source_geography_name", "first")
            if "source_geography_name" in allocated.columns
            else ("source_geography_code", "first"),
        oa_count=("OA21CD", "nunique"),
        source_area_population=("source_area_population", "first"),
        allocation_weight_sum=("allocation_weight", "sum"),
    )
)

allocation_check["allocation_weight_error"] = (
    allocation_check["allocation_weight_sum"] - 1
).abs()

print("Allocated OA rows:", len(allocated))
print("Allocated result areas:", allocated["result_area_key"].nunique())
print("Allocation weight check:")
display(allocation_check["allocation_weight_sum"].describe())

bad_weights = allocation_check[
    allocation_check["allocation_weight_error"] > 0.000001
].copy()

print("Result areas with bad allocation weights:", len(bad_weights))
if len(bad_weights) > 0:
    display(bad_weights.head(20))

display(allocated.head())

Ready result areas: 11744
Ready rows: 11744
Duplicate result_area_key rows: 0
Repeated source geography codes: 3


,source_boundary_year,source_geography_code,result_area_count,rows
1076,2022,E05009317,2,2
2263,2022,E05013778,2,2
2952,2022,W05001146,2,2


Ward-ready result areas: 10856
CED-ready result areas: 888
Allocated OA rows: 327629
Allocated result areas: 11592
Allocation weight check:


count    1.159200e+04
mean     1.000000e+00
std      1.031216e-18
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: allocation_weight_sum, dtype: float64

Result areas with bad allocation weights: 0


,result_area_key,source_year,election_year,election_date,source_file,council_name,lad_code,ward_name,standard_ward_name,ward_code,...,allocated_top_party_votes,allocated_runner_up_party_votes,allocated_con_votes,allocated_lab_votes,allocated_ld_votes,allocated_green_votes,allocated_reform_ukip_brexit_votes,allocated_independent_votes,allocated_sdp_votes,allocated_other_votes
0,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,local-elections-2022.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,35.933383,30.543376,35.933383,30.543376,19.901566,0.0,0.0,0.0,0.0,0.0
1,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,local-elections-2022.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,43.301019,36.805866,43.301019,36.805866,23.982103,0.0,0.0,0.0,0.0,0.0
2,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,local-elections-2022.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,35.416356,30.103903,35.416356,30.103903,19.615213,0.0,0.0,0.0,0.0,0.0
3,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,local-elections-2022.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,45.498384,38.673627,45.498384,38.673627,25.199105,0.0,0.0,0.0,0.0,0.0
4,2022|CODE|ADUR|E05007562|BUCKINGHAM,2022,2022,2022-05-05,local-elections-2022.xlsx,Adur,E07000223,Buckingham,Buckingham,E05007562,...,37.355208,31.751926,37.355208,31.751926,20.689038,0.0,0.0,0.0,0.0,0.0


## 14.7 Diagnostics and unmatched ready rows


In [18]:
allocated_keys = set(allocated["result_area_key"].dropna().unique())
ready_keys = set(ready_areas["result_area_key"].dropna().unique())

unallocated_ready = ready_areas[~ready_areas["result_area_key"].isin(allocated_keys)].copy()

allocation_diagnostics = (
    allocated
    .groupby(["source_year", "source_boundary_year", "detected_geography_type", "allocation_method"], dropna=False, as_index=False)
    .agg(
        result_areas=("result_area_key", "nunique"),
        oa_rows=("OA21CD", "count"),
        allocated_population=("population", "sum"),
    )
)

weight_check = (
    allocated
    .groupby("result_area_key", as_index=False)
    .agg(
        allocation_weight_sum=("allocation_weight", "sum"),
        oa_count=("OA21CD", "nunique"),
        source_area_population=("source_area_population", "first"),
    )
)

print("Ready result areas:", len(ready_keys))
print("Allocated result areas:", len(allocated_keys))
print("Unallocated ready result areas:", len(unallocated_ready))

display(allocation_diagnostics)
display(weight_check["allocation_weight_sum"].describe())
if len(unallocated_ready) > 0:
    display(unallocated_ready[["result_area_key", "source_year", "source_geography_code", "detected_geography_type", "join_readiness_status"]].head(50))


Ready result areas: 11744
Allocated result areas: 11592
Unallocated ready result areas: 152


,source_year,source_boundary_year,detected_geography_type,allocation_method,result_areas,oa_rows,allocated_population
0,2022,2022,electoral_ward_or_division,source_ward_to_oa21_population_weighted,3609,108531,34620268
1,2023,2023,electoral_ward_or_division,source_ward_to_oa21_population_weighted,4831,119086,37356528
2,2024,2024,electoral_ward_or_division,source_ward_to_oa21_population_weighted,1903,63971,20235728
3,2025,2025,county_electoral_division,source_ced_to_oa21_via_ward_population_weighted,736,23521,7331118
4,2025,2025,electoral_ward_or_division,source_ward_to_oa21_population_weighted,513,12520,3893328


count    1.159200e+04
mean     1.000000e+00
std      1.031216e-18
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: allocation_weight_sum, dtype: float64

,result_area_key,source_year,source_geography_code,detected_geography_type,join_readiness_status
14243,2025|CODE|BASSETLAW|E58001221|WORKSOP_WEST,2025,E58001221,county_electoral_division,ced_bridge_available
14262,2025|CODE|BOSTON|E58000938|SKIRBECK,2025,E58000938,county_electoral_division,ced_bridge_available
14273,2025|CODE|BROXBOURNE|E58000620|FLAMSTEAD_END_A...,2025,E58000620,county_electoral_division,ced_bridge_available
14275,2025|CODE|BROXBOURNE|E58000643|HODDESDON_NORTH,2025,E58000643,county_electoral_division,ced_bridge_available
14340,2025|CODE|CAMBRIDGE|E58000050|ABBEY,2025,E58000050,county_electoral_division,ced_bridge_available
14341,2025|CODE|CAMBRIDGE|E58000052|ARBURY,2025,E58000052,county_electoral_division,ced_bridge_available
14342,2025|CODE|CAMBRIDGE|E58000057|CASTLE,2025,E58000057,county_electoral_division,ced_bridge_available
14343,2025|CODE|CAMBRIDGE|E58000059|CHERRY_HINTON,2025,E58000059,county_electoral_division,ced_bridge_available
14344,2025|CODE|CAMBRIDGE|E58000060|CHESTERTON,2025,E58000060,county_electoral_division,ced_bridge_available
14345,2025|CODE|CAMBRIDGE|E58000072|KING_S_HEDGES,2025,E58000072,county_electoral_division,ced_bridge_available


## 14.8 Save allocated OA21 election layer


In [19]:
# Keep a compact but useful order.
id_cols = [
    "result_area_key", "source_year", "election_year", "election_date",
    "council_name", "lad_code", "ward_name", "standard_ward_name", "ward_code",
    "source_boundary_year", "source_geography_code", "detected_geography_type",
    "allocation_method", "OA21CD", "population", "source_area_population", "allocation_weight",
]

allocated_cols = [c for c in allocated.columns if c.startswith("allocated_")]
extra_cols = [
    "top_party_by_votes", "runner_up_party_by_votes",
    "geography_type", "atlas_join_strategy", "join_readiness_status", "data_quality_flag",
]

ordered = [c for c in id_cols + extra_cols + numeric_cols + allocated_cols if c in allocated.columns]
remaining = [c for c in allocated.columns if c not in ordered]

allocated_out = allocated[ordered + remaining].copy()

allocated_path = OUTPUT_DIR / "election_results_oa21_allocated_v1.csv"
diag_path = OUTPUT_DIR / "election_results_oa21_allocation_diagnostics_v1.csv"
unmatched_path = OUTPUT_DIR / "election_result_areas_unallocated_v1.csv"
weight_path = OUTPUT_DIR / "election_results_oa21_weight_check_v1.csv"

allocated_out.to_csv(allocated_path, index=False)
allocation_diagnostics.to_csv(diag_path, index=False)
unallocated_ready.to_csv(unmatched_path, index=False)
weight_check.to_csv(weight_path, index=False)

print("Saved:")
print(" ", allocated_path)
print(" ", diag_path)
print(" ", unmatched_path)
print(" ", weight_path)


Saved:
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\oa21_allocated_v1\election_results_oa21_allocated_v1.csv
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\oa21_allocated_v1\election_results_oa21_allocation_diagnostics_v1.csv
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\oa21_allocated_v1\election_result_areas_unallocated_v1.csv
  c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\oa21_allocated_v1\election_results_oa21_weight_check_v1.csv


## 14.9 Next step

Notebook 15 will merge this OA-level election allocation with the K7 OA atlas, aggregate it to WD25/LAD25/cluster geographies, and join the latest electoral metrics back to the ward atlas.


In [20]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = PROJECT_DIR / "data" / "processed" / "election_results" / "oa21_allocated_v1"

allocated_path = OUTPUT_DIR / "election_results_oa21_allocated_v1.csv"
weight_path = OUTPUT_DIR / "election_results_oa21_weight_check_v1.csv"
diag_path = OUTPUT_DIR / "election_results_oa21_allocation_diagnostics_v1.csv"
unallocated_path = OUTPUT_DIR / "election_result_areas_unallocated_v1.csv"

allocated = pd.read_csv(allocated_path, low_memory=False)
weights = pd.read_csv(weight_path, low_memory=False)
diag = pd.read_csv(diag_path, low_memory=False)
unallocated = pd.read_csv(unallocated_path, low_memory=False)

print("Allocated OA rows:", len(allocated))
print("Allocated result areas:", allocated["result_area_key"].nunique())
print("Weight check rows:", len(weights))
print("Unallocated result areas:", len(unallocated))

display(diag)

print("Weight sum min:", weights["allocation_weight_sum"].min())
print("Weight sum max:", weights["allocation_weight_sum"].max())
print("Bad weights:", (weights["allocation_weight_sum"].sub(1).abs() > 0.000001).sum())

Allocated OA rows: 327629
Allocated result areas: 11592
Weight check rows: 11592
Unallocated result areas: 152


,source_year,source_boundary_year,detected_geography_type,allocation_method,result_areas,oa_rows,allocated_population
0,2022,2022,electoral_ward_or_division,source_ward_to_oa21_population_weighted,3609,108531,34620268
1,2023,2023,electoral_ward_or_division,source_ward_to_oa21_population_weighted,4831,119086,37356528
2,2024,2024,electoral_ward_or_division,source_ward_to_oa21_population_weighted,1903,63971,20235728
3,2025,2025,county_electoral_division,source_ced_to_oa21_via_ward_population_weighted,736,23521,7331118
4,2025,2025,electoral_ward_or_division,source_ward_to_oa21_population_weighted,513,12520,3893328


Weight sum min: 1.0
Weight sum max: 1.0
Bad weights: 0
